# Tragedy: "Epicness" by Character

In [1]:
import numpy as np
import pandas as pd

homer_df = pd.read_parquet("../parquet/homer_dtm.parquet")
tragedy_df = pd.read_parquet("../parquet/tragedy_dtm.parquet")
loglikelihood_df = pd.read_parquet("../parquet/epic_tragic_lemmata_loglikelihood.parquet")

In [ ]:
ll = loglikelihood_df.set_index("lemma")["log_likelihood"]
speaker_avg = tragedy_df.multiply(ll, axis=0).sum() / tragedy_df.sum()
speaker_df = speaker_avg.reset_index()
speaker_df.columns = ["dramatist", "title", "speaker", "avg_log_likelihood"]

In [22]:
speaker_df

,dramatist,title,speaker,avg_log_likelihood
0,Aeschylus,Agamemnon,Αἴγισθος,-41.018516
1,Aeschylus,Agamemnon,Κασάνδρα,-55.930683
2,Aeschylus,Agamemnon,Κλυταιμήστρα,-34.180206
3,Aeschylus,Agamemnon,Κῆρυξ,-40.723311
4,Aeschylus,Agamemnon,Φύλαξ,-19.994088
...,...,...,...,...
297,Sophocles,Trachiniae,Ἡμιχόριον,-9.644201
298,Sophocles,Trachiniae,Ἡμιχόριον 1,-42.311606
299,Sophocles,Trachiniae,Ἡμιχόριον 2,-21.134394
300,Sophocles,Trachiniae,Ἡρακλῆς,-70.303616


In [27]:
import altair as alt

speaker_df["label"] = speaker_df["dramatist"] + " – " + speaker_df["title"] + " – " + speaker_df["speaker"]

alt.Chart(speaker_df).mark_point().encode(
    x=alt.X("avg_log_likelihood:Q", title="Avg log-likelihood"),
    y=alt.Y("label:N", sort="-x", title=None, axis=alt.Axis(labelLimit=300)),
    color="dramatist:N"
).properties(width=600, height=3000)

alt.Chart(...)

In [ ]:
import re

def classify(speaker):
    s = speaker.strip()
    if re.match(r"(Χορ|Ἡμιχ|Ἡμίχο|Προπομπ|Ημιχ)", s):
        return "chorus"
    if re.match(r"(Ἄγγελος|Ἐξάγγελος|Κῆρυξ|Λίχας|Ταλθύβιος|Αγγελος)", s):
        return "messenger"
    return "other"

speaker_df["speaker_class"] = speaker_df["speaker"].apply(classify)

In [28]:
ll = loglikelihood_df.set_index("lemma")["log_likelihood"]

def classify(speaker):
    s = speaker.strip()
    if re.match(r"(Χορ|Ἡμιχ|Ἡμίχο|Προπομπ|Ημιχ)", s):
        return "chorus"
    if re.match(r"(Ἄγγελος|Ἐξάγγελος|Κῆρυξ|Λίχας|Ταλθύβιος|Αγγελος)", s):
        return "messenger"
    return "other"

class_counts = tragedy_df.T
class_counts["class"] = class_counts.index.get_level_values("speaker").map(classify)
class_counts = class_counts.groupby("class").sum().T

class_avg = class_counts.multiply(ll, axis=0).sum() / class_counts.sum()
class_avg = class_avg.reset_index(name="avg_log_likelihood")

In [29]:
alt.Chart(class_avg).mark_point().encode(
    x=alt.X("avg_log_likelihood:Q"),
    y=alt.Y("class:N", sort="-x", title=None),
    color="class:N",
).properties(width=600, height=200)

alt.Chart(...)